# Data Cleaning

In [ ]:
import re
import numpy as np
import pandas as pd

# Load datasets
df_structured = pd.read_csv("structured_data.csv")
df_text = pd.read_csv("exit_interviews.csv")

In [ ]:
import pandas as pd

# Load files
df_struct = pd.read_csv("structured_data.csv")
df_text = pd.read_csv("exit_interviews.csv")

print("=== STRUCTURED DATA DIAGNOSTIC ===")
print("\n--- Missing Values Count ---")
print(df_struct.isnull().sum())

print("\n--- Unique Categorical Values (Check for Typos/Variations) ---")
for col in [
    "Gender",
    "Region",
    "Subscription_Type",
    "Churn_Flag",
    "Support_Tickets",
]:
  if col in df_struct.columns:
    print(f"\n{col}:")
    print(df_struct[col].dropna().unique()[:30])

print("\n--- Numeric Ranges (Check for Outliers/Negative Values) ---")
print(df_struct[["Age", "Tenure_Months", "Monthly_Spend"]].describe())

print("\n\n=== UNSTRUCTURED DATA DIAGNOSTIC ===")
print("\n--- Missing Text Count ---")
print(df_text.isnull().sum())

if "Churn_Flag" in df_text.columns:
  print("\nText File Churn_Flag Values:")
  print(df_text["Churn_Flag"].unique())

=== STRUCTURED DATA DIAGNOSTIC ===

--- Missing Values Count ---
Customer_ID            0
Age                    0
Gender               342
Region               229
Tenure_Months          0
Subscription_Type    239
Monthly_Spend          0
Support_Tickets        0
Churn_Flag           586
Signup_Date            0
dtype: int64

--- Unique Categorical Values (Check for Typos/Variations) ---

Gender:
['Female' 'M' 'Male' 'Non-binary' 'male' ' Male  ' ' F  ' 'F' 'MALE '
 'female' ' m ' 'FEMALE' 'nonbinary' ' fem ' 'na' 'Prefer not to say'
 ' Female  ' 'NB' ' MALE   ' ' female  ' 'Non Binary' '-' ' Non-binary  '
 ' male  ' ' Prefer not to say  ' ' FEMALE  ' '  m   ' '  fem   ' ' M  '
 'Prefer-not-to-say']

Region:
['South' 'Central' 'North' 'East' 'West' 'Ctr' 'Nrt' ' South' 'Sth'
 '   West ' 'Est' 'EAST' ' East  ' 'North  ' ' Central  ' 'CENTRAL'
 'NORTH' 'Wst' 'WEST' 'Central-' ' South  ' ' CENTRAL  ' ' EAST  ' 'SOUTH'
 ' North    ' ' SOUTH  ' ' Ctr  ' ' North  ' ' West  ' '-']

Subscript

In [ ]:
# =========================================================
# 1. CLEAN STRUCTURED DATA
# =========================================================

# Standardize Customer_ID
df_struct["Customer_ID"] = df_struct["Customer_ID"].astype(str).str.upper()

# Clean Gender: Map all 30 variations into 4 clean categories
gender_clean = {
    "FEMALE": "Female",
    "FEM": "Female",
    "F": "Female",
    "MALE": "Male",
    "M": "Male",
    "NON-BINARY": "Non-binary",
    "NONBINARY": "Non-binary",
    "NON BINARY": "Non-binary",
    "NB": "Non-binary",
}


def clean_gender(val):
  if pd.isna(val) or str(val).strip().lower() in ["na", "-", ""]:
    return "Prefer not to say"
  clean_val = str(val).strip().upper()
  if clean_val in gender_clean:
    return gender_clean[clean_val]
  if "PREFER" in clean_val:
    return "Prefer not to say"
  return "Prefer not to say"


df_struct["Gender"] = df_struct["Gender"].apply(clean_gender)

# Clean Region: Fix typos (Ctr, Nrt, Sth, Est, Wst)
region_clean = {
    "CTR": "Central",
    "CENTRAL-": "Central",
    "CENTRAL": "Central",
    "NRT": "North",
    "NORTH": "North",
    "STH": "South",
    "SOUTH": "South",
    "EST": "East",
    "EAST": "East",
    "WST": "West",
    "WEST": "West",
}


def clean_region(val):
  if pd.isna(val) or str(val).strip() == "-":
    return "Unknown"
  clean_val = str(val).strip().upper()
  return region_clean.get(clean_val, "Unknown")


df_struct["Region"] = df_struct["Region"].apply(clean_region)

# Clean Subscription_Type: Map synonyms (entry, mid, pro, gold, corp, STD, PRM, ENT)
sub_clean = {
    "ENTRY": "Basic",
    "STARTER": "Basic",
    "BASIC": "Basic",
    "MID": "Standard",
    "STD": "Standard",
    "STANDARD": "Standard",
    "PRO": "Premium",
    "GOLD": "Premium",
    "PRM": "Premium",
    "PREMIUM": "Premium",
    "CORP": "Enterprise",
    "CORPORATE": "Enterprise",
    "ENT": "Enterprise",
    "ENTERPRISE": "Enterprise",
}


def clean_sub(val):
  if pd.isna(val) or str(val).strip() == "-":
    return "Unknown"
  clean_val = str(val).strip().upper()
  return sub_clean.get(clean_val, "Unknown")


df_struct["Subscription_Type"] = df_struct["Subscription_Type"].apply(clean_sub)

# Clean Monthly_Spend: Strip currency symbols & corrupt characters (e.g. â,’)
df_struct["Monthly_Spend"] = (
    df_struct["Monthly_Spend"].astype(str).str.replace(r"[^\d.]", "", regex=True)
)
df_struct["Monthly_Spend"] = pd.to_numeric(
    df_struct["Monthly_Spend"], errors="coerce"
)
df_struct["Monthly_Spend"] = df_struct["Monthly_Spend"].fillna(
    df_struct["Monthly_Spend"].median()
)

# Clean Support_Tickets: Turn negative ticket counts into positive numbers
df_struct["Support_Tickets"] = df_struct["Support_Tickets"].abs()

# Clean Age & Tenure_Months: Fix extreme outliers and negative values
# - Fix Age (clip under 18 or over 100 to median)
median_age = df_struct[
    (df_struct["Age"] >= 18) & (df_struct["Age"] <= 100)
]["Age"].median()
df_struct.loc[
    (df_struct["Age"] < 18) | (df_struct["Age"] > 100), "Age"
] = median_age

# - Fix Tenure_Months (convert negative months to positive)
df_struct["Tenure_Months"] = df_struct["Tenure_Months"].abs()

# Clean Churn_Flag: Standardize to 1 / 0
churn_map = {"1": 1, "YES": 1, "0": 0, "NO": 0, 1: 1, 0: 0}
df_struct["Churn_Flag"] = (
    df_struct["Churn_Flag"].astype(str).str.strip().str.upper().map(churn_map)
)

# Deduplicate by Customer_ID
df_struct = df_struct.drop_duplicates(subset=["Customer_ID"])

In [ ]:
# =========================================================
# 2. CLEAN UNSTRUCTURED DATA (Exit Interviews)
# =========================================================

# Standardize Customer_ID
df_text["Customer_ID"] = df_text["Customer_ID"].astype(str).str.upper()

# Standardize Churn_Flag in text file
df_text["Churn_Flag"] = (
    df_text["Churn_Flag"].astype(str).str.strip().str.upper().map(churn_map)
)


# Clean Exit_Reason_Text: Remove HTML, special symbols, and null placeholders
def clean_text(text):
  if pd.isna(text) or not isinstance(text, str):
    return np.nan
  # Remove HTML tags (<p>, <br>, etc.)
  text = re.sub(r"<[^>]+>", "", text)
  # Normalize spacing
  text = re.sub(r"\s+", " ", text).strip()
  # Remove null placeholders
  if text.lower() in ["n/a", "none", "null", "no comment", "-", ""] or len(
      text
  ) < 5:
    return np.nan
  return text


df_text["Exit_Reason_Text"] = df_text["Exit_Reason_Text"].apply(clean_text)

# Drop rows missing text feedback
df_text = df_text.dropna(subset=["Exit_Reason_Text"]).reset_index(drop=True)

In [ ]:
# =========================================================
# 3. SAVE CLEANED FILES
# =========================================================
df_struct.to_csv("cleaned_structured_data.csv", index=False)
df_text.to_csv("cleaned_exit_interviews.csv", index=False)

print("✅ Data cleaning complete!")
print(
    f"Structured Data Rows: {len(df_struct)} | Missing Churn Flags:"
    f" {df_struct['Churn_Flag'].isnull().sum()}"
)
print(f"Unstructured Data Rows: {len(df_text)}")

✅ Data cleaning complete!
Structured Data Rows: 95000 | Missing Churn Flags: 95000
Unstructured Data Rows: 3473


In [ ]:
import pandas as pd

# Load the newly saved cleaned datasets
df_struct_clean = pd.read_csv("cleaned_structured_data.csv")
df_text_clean = pd.read_csv("cleaned_exit_interviews.csv")

print("==========================================")
print("     STRUCTURED DATA VERIFICATION        ")
print("==========================================")

# 1. Check for unexpected missing values
print("--- Missing Values ---")
print(df_struct_clean.isnull().sum())

# 2. Check unique standardized categories
print("\n--- Standardized Categories ---")
print("Gender:", df_struct_clean["Gender"].unique())
print("Region:", df_struct_clean["Region"].unique())
print(
    "Subscription_Type:", df_struct_clean["Subscription_Type"].unique()
)
print("Churn_Flag:", df_struct_clean["Churn_Flag"].unique())

# 3. Check numeric boundaries (Age, Tenure, Spend, Tickets)
print("\n--- Numeric Ranges Check ---")
print(
    df_struct_clean[
        ["Age", "Tenure_Months", "Monthly_Spend", "Support_Tickets"]
    ].describe()
)

# 4. Check for duplicates
print("\n--- Duplicate Customer_IDs ---")
print("Duplicates Count:", df_struct_clean["Customer_ID"].duplicated().sum())

print("\n==========================================")
print("    UNSTRUCTURED DATA VERIFICATION       ")
print("==========================================")

# 1. Check missing values in text data
print("--- Missing Text Count ---")
print(df_text_clean.isnull().sum())

# 2. Sample cleaned feedback to confirm HTML tags & junk are removed
print("\n--- Sample Cleaned Feedback (First 3 rows) ---")
for idx, text in enumerate(df_text_clean["Exit_Reason_Text"].head(3), 1):
  print(f"{idx}. {text[:120]}...")

     STRUCTURED DATA VERIFICATION        
--- Missing Values ---
Customer_ID              0
Age                      0
Gender                   0
Region                   0
Tenure_Months            0
Subscription_Type        0
Monthly_Spend            0
Support_Tickets          0
Churn_Flag           95000
Signup_Date              0
dtype: int64

--- Standardized Categories ---
Gender: ['Female' 'Male' 'Non-binary' 'Prefer not to say']
Region: ['South' 'Central' 'North' 'East' 'West' 'Unknown']
Subscription_Type: ['Standard' 'Basic' 'Enterprise' 'Premium' 'Unknown']
Churn_Flag: [nan]

--- Numeric Ranges Check ---
                Age  Tenure_Months  Monthly_Spend  Support_Tickets
count  95000.000000   95000.000000   95000.000000     95000.000000
mean      34.254463      60.991716      39.435078         1.200389
std        9.514929      36.198504      39.437650         1.095980
min       18.000000       1.000000       5.000000         0.000000
25%       27.000000      31.000000      15.5

In [ ]:
import numpy as np
import pandas as pd

# Load raw structured dataset again
df_struct = pd.read_csv("structured_data.csv")
df_clean = pd.read_csv("cleaned_structured_data.csv")


# Clean Churn_Flag properly by handling float strings ('0.0', '1.0')
def fix_churn(val):
  if pd.isna(val):
    return np.nan
  val_str = str(val).strip().upper()
  if val_str in ["1", "1.0", "YES"]:
    return 1
  if val_str in ["0", "0.0", "NO"]:
    return 0
  return np.nan


df_clean["Churn_Flag"] = df_struct["Churn_Flag"].apply(fix_churn)

# Also fix the 20 missing Churn_Flags in unstructured dataset if needed
df_text = pd.read_csv("cleaned_exit_interviews.csv")
df_text["Churn_Flag"] = df_text["Churn_Flag"].apply(fix_churn)

# Overwrite cleaned CSVs
df_clean.to_csv("cleaned_structured_data.csv", index=False)
df_text.to_csv("cleaned_exit_interviews.csv", index=False)

print("Updated Churn_Flag value counts in Structured Data:")
print(df_clean["Churn_Flag"].value_counts(dropna=False))

Updated Churn_Flag value counts in Structured Data:
Churn_Flag
0.0    74679
1.0    19740
NaN      581
Name: count, dtype: int64


# AI Classification & Mood Analysis

In [ ]:
!pip install google-genai

In [ ]:
import pandas as pd

# Load both cleaned datasets
df_struct = pd.read_csv("cleaned_structured_data.csv")
df_text = pd.read_csv("classified_exit_interviews.csv")

print("==========================================")
print("     CLASSIFIED DATA VALIDATION CHECK     ")
print("==========================================")

# 1. Check Missing Values in Classification Columns
print("--- 1. Missing Values Count ---")
print(df_text[["Exit_Reason", "Customer_Mood"]].isnull().sum())

# 2. Validate Allowed Categories
valid_reasons = {
    "Price",
    "Support",
    "Product",
    "Performance",
    "Onboarding",
    "Competitor",
    "Value",
}
valid_moods = {
    "Angry",
    "Frustrated",
    "Disappointed",
    "Neutral",
    "Hopeful",
    "Positive",
}

found_reasons = set(df_text["Exit_Reason"].dropna().unique())
found_moods = set(df_text["Customer_Mood"].dropna().unique())

print("\n--- 2. Category Compliance Check ---")
print("Exit Reasons:", df_text["Exit_Reason"].unique())
print("Invalid Reasons:", found_reasons - valid_reasons or "None ✅")

print("\nCustomer Moods:", df_text["Customer_Mood"].unique())
print("Invalid Moods:", found_moods - valid_moods or "None ✅")

# 3. Check Merge Compatibility with Structured Data
matching_ids = set(df_text["Customer_ID"]).issubset(set(df_struct["Customer_ID"]))
print("\n--- 3. ID Match with Structured Data ---")
print(
    f"Total Text Rows: {len(df_text)} | All Customer_IDs exist in structured"
    f" data: {matching_ids}"
)

# 4. Preview Reason x Mood Cross-tabulation
print("\n--- 4. Exit Reason x Mood Matrix Preview ---")
print(pd.crosstab(df_text["Exit_Reason"], df_text["Customer_Mood"]))

     CLASSIFIED DATA VALIDATION CHECK     
--- 1. Missing Values Count ---
Exit_Reason      0
Customer_Mood    0
dtype: int64

--- 2. Category Compliance Check ---
Exit Reasons: ['Performance' 'Price' 'Support' 'Value' 'Competitor' 'Onboarding'
 'Product']
Invalid Reasons: None ✅

Customer Moods: ['Disappointed' 'Positive' 'Hopeful' 'Frustrated' 'Neutral']
Invalid Moods: None ✅

--- 3. ID Match with Structured Data ---
Total Text Rows: 3473 | All Customer_IDs exist in structured data: False

--- 4. Exit Reason x Mood Matrix Preview ---
Customer_Mood  Disappointed  Frustrated  Hopeful  Neutral  Positive
Exit_Reason                                                        
Competitor              346           0       15        0        16
Onboarding               15           4        1        1        28
Performance             347           3        1        0        17
Price                  1104           0       11        0        23
Product                 370           1        8  

# EDA (Exploratory Data analysis)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Load datasets
df_struct = pd.read_csv("cleaned_structured_data.csv")
df_text = pd.read_csv("classified_exit_interviews.csv")

# Ensure exact string formatting on Customer_ID for clean merging
df_struct["Customer_ID"] = df_struct["Customer_ID"].astype(str).str.strip()
df_text["Customer_ID"] = df_text["Customer_ID"].astype(str).str.strip()

# Merge structured and unstructured data
df_master = pd.merge(df_struct, df_text, on="Customer_ID", how="left")
df_master["Churn_Flag"] = pd.to_numeric(
    df_master["Churn_Flag_x"], errors="coerce"
)

# ---------------------------------------------------------
# 1. STRUCTURED DATA METRICS
# ---------------------------------------------------------
print("=== 1. OVERALL CHURN METRICS ===")
total_customers = len(df_master)
churned_customers = df_master["Churn_Flag"].sum()
overall_churn_rate = (churned_customers / total_customers) * 100
print(f"Total Customers: {total_customers:,}")
print(f"Churned Customers: {int(churned_customers):,}")
print(f"Overall Churn Rate: {overall_churn_rate:.2f}%\n")

print("=== 2. CHURN RATE BY SUBSCRIPTION TYPE ===")
sub_churn = (
    df_master.groupby("Subscription_Type")["Churn_Flag"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
sub_churn["mean"] = sub_churn["mean"] * 100
sub_churn.columns = [
    "Subscription_Type",
    "Total_Customers",
    "Churned_Count",
    "Churn_Rate_%",
]
print(sub_churn.sort_values(by="Churn_Rate_%", ascending=False).to_string(
    index=False
))

print("\n=== 3. CHURN RATE BY REGION ===")
region_churn = (
    df_master.groupby("Region")["Churn_Flag"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
region_churn["mean"] = region_churn["mean"] * 100
region_churn.columns = [
    "Region",
    "Total_Customers",
    "Churned_Count",
    "Churn_Rate_%",
]
print(
    region_churn.sort_values(by="Churn_Rate_%", ascending=False).to_string(
        index=False
    )
)

print("\n=== 4. SUPPORT TICKETS VS CHURN RATE ===")
ticket_churn = (
    df_master.groupby("Support_Tickets")["Churn_Flag"]
    .agg(["count", "mean"])
    .reset_index()
)
ticket_churn["mean"] = ticket_churn["mean"] * 100
ticket_churn.columns = ["Support_Tickets", "Total_Customers", "Churn_Rate_%"]
print(ticket_churn.to_string(index=False))

# ---------------------------------------------------------
# 2. UNSTRUCTURED TEXT METRICS
# ---------------------------------------------------------
print("\n=== 5. TOP EXIT REASONS ===")
print(df_text["Exit_Reason"].value_counts().to_string())

print("\n=== 6. CUSTOMER MOOD DISTRIBUTION ===")
print(df_text["Customer_Mood"].value_counts().to_string())

print("\n=== 7. EXIT REASON x MOOD CROSSTAB MATRIX ===")
matrix = pd.crosstab(
    df_text["Exit_Reason"], df_text["Customer_Mood"], margins=True
)
print(matrix.to_string())

# ---------------------------------------------------------
# 3. SAVE MERGED MASTER FILE FOR POWER BI
# ---------------------------------------------------------
df_master.to_csv("final_merged_data.csv", index=False)
print("\n✅ Saved 'final_merged_data.csv' for Power BI dashboard import!")

=== 1. OVERALL CHURN METRICS ===
Total Customers: 95,090
Churned Customers: 0
Overall Churn Rate: 0.00%

=== 2. CHURN RATE BY SUBSCRIPTION TYPE ===
Subscription_Type  Total_Customers  Churned_Count  Churn_Rate_%
            Basic                0            0.0           NaN
       Enterprise                0            0.0           NaN
          Premium                0            0.0           NaN
         Standard                0            0.0           NaN
          Unknown                0            0.0           NaN

=== 3. CHURN RATE BY REGION ===
 Region  Total_Customers  Churned_Count  Churn_Rate_%
Central                0            0.0           NaN
   East                0            0.0           NaN
  North                0            0.0           NaN
  South                0            0.0           NaN
Unknown                0            0.0           NaN
   West                0            0.0           NaN

=== 4. SUPPORT TICKETS VS CHURN RATE ===
 Support_Ticket

In [ ]:
import pandas as pd

df_struct = pd.read_csv("cleaned_structured_data.csv")
print(df_struct["Churn_Flag"].value_counts(dropna=False))

Churn_Flag
NaN    95000
Name: count, dtype: int64


In [ ]:
import pandas as pd

# 1. Load original structured data to ensure raw churn values are preserved
df_raw = pd.read_csv("structured_data.csv")
df_clean = pd.read_csv("cleaned_structured_data.csv")
df_text = pd.read_csv("classified_exit_interviews.csv")


# Clean Churn_Flag properly
def parse_churn(val):
  if pd.isna(val):
    return 0
  val_str = str(val).strip().upper()
  if val_str in ["1", "1.0", "YES"]:
    return 1
  return 0


# Force clean Churn_Flag back onto structured data
df_clean["Churn_Flag"] = df_raw["Churn_Flag"].apply(parse_churn)
df_clean["Customer_ID"] = df_clean["Customer_ID"].astype(str).str.strip()

# Clean text dataset
df_text["Customer_ID"] = df_text["Customer_ID"].astype(str).str.strip()
if "Churn_Flag" in df_text.columns:
  df_text = df_text.drop(columns=["Churn_Flag"])

# 2. Merge Structured and AI Text Data
df_master = pd.merge(df_clean, df_text, on="Customer_ID", how="left")

# ---------------------------------------------------------
# 3. COMPUTING FINAL EDA METRICS
# ---------------------------------------------------------
print("=== 1. OVERALL CHURN METRICS ===")
total_cust = len(df_master)
churn_cust = df_master["Churn_Flag"].sum()
print(f"Total Customers: {total_cust:,}")
print(f"Churned Customers: {churn_cust:,}")
print(f"Overall Churn Rate: {(churn_cust / total_cust) * 100:.2f}%\n")

print("=== 2. CHURN RATE BY SUBSCRIPTION TYPE ===")
sub_eda = (
    df_master.groupby("Subscription_Type")["Churn_Flag"]
    .agg(
        Total_Customers="count", Churned_Count="sum", Churn_Rate_="mean"
    )
    .reset_index()
)
sub_eda["Churn_Rate_%"] = (sub_eda["Churn_Rate_"] * 100).round(2)
print(
    sub_eda[
        ["Subscription_Type", "Total_Customers", "Churned_Count", "Churn_Rate_%"]
    ]
    .sort_values(by="Churn_Rate_%", ascending=False)
    .to_string(index=False)
)

print("\n=== 3. CHURN RATE BY REGION ===")
reg_eda = (
    df_master.groupby("Region")["Churn_Flag"]
    .agg(
        Total_Customers="count", Churned_Count="sum", Churn_Rate_="mean"
    )
    .reset_index()
)
reg_eda["Churn_Rate_%"] = (reg_eda["Churn_Rate_"] * 100).round(2)
print(
    reg_eda[["Region", "Total_Customers", "Churned_Count", "Churn_Rate_%"]]
    .sort_values(by="Churn_Rate_%", ascending=False)
    .to_string(index=False)
)

print("\n=== 4. SUPPORT TICKETS VS CHURN RATE ===")
tkt_eda = (
    df_master.groupby("Support_Tickets")["Churn_Flag"]
    .agg(
        Total_Customers="count", Churned_Count="sum", Churn_Rate_="mean"
    )
    .reset_index()
)
tkt_eda["Churn_Rate_%"] = (tkt_eda["Churn_Rate_"] * 100).round(2)
print(
    tkt_eda[
        ["Support_Tickets", "Total_Customers", "Churned_Count", "Churn_Rate_%"]
    ].to_string(index=False)
)

# 4. Save Final Master CSV
df_master.to_csv("final_merged_data.csv", index=False)
print("\n✅ Overwritten 'final_merged_data.csv' successfully!")

=== 1. OVERALL CHURN METRICS ===
Total Customers: 95,090
Churned Customers: 19,811
Overall Churn Rate: 20.83%

=== 2. CHURN RATE BY SUBSCRIPTION TYPE ===
Subscription_Type  Total_Customers  Churned_Count  Churn_Rate_%
         Standard            31472           6981         22.18
            Basic            33026           7318         22.16
          Unknown              313             64         20.45
          Premium            25397           4712         18.55
       Enterprise             4882            736         15.08

=== 3. CHURN RATE BY REGION ===
 Region  Total_Customers  Churned_Count  Churn_Rate_%
  South            18853           3994         21.18
  North            20884           4408         21.11
   West            19018           3926         20.64
   East            20891           4312         20.64
Central            15131           3107         20.53
Unknown              313             64         20.45

=== 4. SUPPORT TICKETS VS CHURN RATE ===
 Support_

In [ ]:
from google.colab import files

# Trigger direct browser download of the merged dataset
files.download("final_merged_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>